In [1]:
import pandas as pd
import numpy as np

matches = pd.read_csv('../data/processed/matches_clean.csv')
deliveries = pd.read_csv('../data/processed/deliveries_clean.csv')
features = pd.read_csv('../data/processed/features.csv')

print("Matches:", matches.shape)
print("Deliveries:", deliveries.shape)

Matches: (1090, 21)
Deliveries: (260920, 17)


In [2]:
team_season_wins = matches.groupby(['season', 'winner']).size().reset_index(name='wins')
team_season_wins.columns = ['Season', 'Team', 'Wins']
team_season_wins.to_csv('../data/processed/pbi_team_season_wins.csv', index=False)
print("Saved pbi_team_season_wins.csv")
print(team_season_wins.head())

Saved pbi_team_season_wins.csv
   Season                   Team  Wins
0    2008    Chennai Super Kings     9
1    2008        Deccan Chargers     2
2    2008         Delhi Capitals     7
3    2008  Kolkata Knight Riders     6
4    2008         Mumbai Indians     7


In [3]:
# Total matches played
team1_matches = matches.groupby('team1').size().reset_index(name='matches_as_team1')
team2_matches = matches.groupby('team2').size().reset_index(name='matches_as_team2')
total_wins = matches.groupby('winner').size().reset_index(name='total_wins')

all_teams = pd.DataFrame({'Team': matches['team1'].unique()})
all_teams = all_teams.merge(team1_matches.rename(columns={'team1':'Team'}), on='Team', how='left')
all_teams = all_teams.merge(team2_matches.rename(columns={'team2':'Team'}), on='Team', how='left')
all_teams = all_teams.merge(total_wins.rename(columns={'winner':'Team'}), on='Team', how='left')

all_teams['total_matches'] = all_teams['matches_as_team1'].fillna(0) + all_teams['matches_as_team2'].fillna(0)
all_teams['total_wins'] = all_teams['total_wins'].fillna(0)
all_teams['win_percentage'] = round(all_teams['total_wins'] / all_teams['total_matches'] * 100, 1)
all_teams = all_teams[['Team', 'total_matches', 'total_wins', 'win_percentage']].sort_values('total_wins', ascending=False)

all_teams.to_csv('../data/processed/pbi_team_performance.csv', index=False)
print("Saved pbi_team_performance.csv")
print(all_teams)

Saved pbi_team_performance.csv
                           Team  total_matches  total_wins  win_percentage
3                Mumbai Indians            261         144            55.2
7           Chennai Super Kings            237         138            58.2
4         Kolkata Knight Riders            251         131            52.2
0   Royal Challengers Bengaluru            252         123            48.8
2                Delhi Capitals            250         115            46.0
5              Rajasthan Royals            219         112            51.1
1                  Punjab Kings            246         112            45.5
10          Sunrisers Hyderabad            182          88            48.4
6               Deccan Chargers             75          29            38.7
14               Gujarat Titans             45          28            62.2
13         Lucknow Super Giants             43          24            55.8
12      Rising Pune Supergiants             30          15           

In [4]:
venue_stats = matches.groupby(['venue', 'winner']).size().reset_index(name='wins')
venue_total = matches.groupby('venue').size().reset_index(name='total_matches')
venue_stats = venue_stats.merge(venue_total, on='venue')
venue_stats['win_pct'] = round(venue_stats['wins'] / venue_stats['total_matches'] * 100, 1)
venue_stats.columns = ['Venue', 'Team', 'Wins', 'Total_Matches', 'Win_Pct']

venue_stats.to_csv('../data/processed/pbi_venue_analysis.csv', index=False)
print("Saved pbi_venue_analysis.csv")
print(venue_stats.head(10))

Saved pbi_venue_analysis.csv
                         Venue                         Team  Wins  \
0         Arun Jaitley Stadium          Chennai Super Kings     1   
1         Arun Jaitley Stadium               Delhi Capitals     8   
2         Arun Jaitley Stadium               Mumbai Indians     1   
3         Arun Jaitley Stadium                 Punjab Kings     1   
4         Arun Jaitley Stadium  Royal Challengers Bengaluru     1   
5         Arun Jaitley Stadium          Sunrisers Hyderabad     2   
6  Arun Jaitley Stadium, Delhi          Chennai Super Kings     2   
7  Arun Jaitley Stadium, Delhi               Delhi Capitals     6   
8  Arun Jaitley Stadium, Delhi               Gujarat Titans     1   
9  Arun Jaitley Stadium, Delhi               Mumbai Indians     3   

   Total_Matches  Win_Pct  
0             14      7.1  
1             14     57.1  
2             14      7.1  
3             14      7.1  
4             14      7.1  
5             14     14.3  
6             1

In [5]:
toss = matches.copy()
toss['toss_won_match'] = (toss['toss_winner'] == toss['winner']).astype(int)

toss_summary = toss.groupby('toss_decision').agg(
    total_matches=('id', 'count'),
    toss_won_match=('toss_won_match', 'sum')
).reset_index()
toss_summary['toss_lost_match'] = toss_summary['total_matches'] - toss_summary['toss_won_match']
toss_summary['win_pct'] = round(toss_summary['toss_won_match'] / toss_summary['total_matches'] * 100, 1)
toss_summary.columns = ['Toss_Decision', 'Total_Matches', 'Won_Match', 'Lost_Match', 'Win_Pct']

toss_summary.to_csv('../data/processed/pbi_toss_analysis.csv', index=False)
print("Saved pbi_toss_analysis.csv")
print(toss_summary)

Saved pbi_toss_analysis.csv
  Toss_Decision  Total_Matches  Won_Match  Lost_Match  Win_Pct
0           bat            390        177         213     45.4
1         field            700        377         323     53.9


In [6]:
season_summary = matches.groupby('season').agg(
    total_matches=('id', 'count'),
    unique_teams=('team1', 'nunique'),
    avg_result_margin=('result_margin', 'mean')
).reset_index()
season_summary['avg_result_margin'] = round(season_summary['avg_result_margin'], 1)
season_summary.columns = ['Season', 'Total_Matches', 'Unique_Teams', 'Avg_Result_Margin']

season_summary.to_csv('../data/processed/pbi_season_summary.csv', index=False)
print("Saved pbi_season_summary.csv")
print(season_summary)

Saved pbi_season_summary.csv
    Season  Total_Matches  Unique_Teams  Avg_Result_Margin
0     2008             58             8               16.0
1     2009             57             7               16.9
2     2010             60             8               19.8
3     2011             72            10               18.9
4     2012             74             9               16.2
5     2013             76             9               19.8
6     2014             60             8               14.7
7     2015             57             8               17.8
8     2016             60             8               15.3
9     2017             59             8               17.1
10    2018             60             8               14.4
11    2019             59             8               15.2
12    2020             60             8               22.6
13    2021             60             8               13.6
14    2022             74            10               17.0
15    2023             73  

In [7]:
feature_importance = pd.DataFrame({
    'Feature': ['Head-to-Head Win Rate', 'Venue Win Rate', 'Recent Form (last 5)', 'Won Toss', 'Toss Decision (Bat)'],
    'Importance_Pct': [43.7, 32.7, 15.5, 4.8, 3.1]
})
feature_importance.to_csv('../data/processed/pbi_feature_importance.csv', index=False)
print("Saved pbi_feature_importance.csv")
print(feature_importance)

Saved pbi_feature_importance.csv
                 Feature  Importance_Pct
0  Head-to-Head Win Rate            43.7
1         Venue Win Rate            32.7
2   Recent Form (last 5)            15.5
3               Won Toss             4.8
4    Toss Decision (Bat)             3.1
